# Notebook 00: MCTS-HNM Master End-to-End Pipeline Orchestrator

This master notebook orchestrates the complete research workflow of **MCTS-HNM** from scratch on remote JupyterHub:
1. **Stage 1: Setup & Data Verification** (deterministic seeding, CIFAR-100 taxonomy, embedding cache check)
2. **Stage 2: Dual Hierarchy Tree Construction** (Native Semantic Tree vs. Ward Confusion Tree)
3. **Stage 3: Tier-1 MCTS Hard Negative Mining** (MLP head on frozen ResNet-18 embeddings)
4. **Stage 4: Tier-2 End-to-End CNN Vision Pipeline** (Trainable ResNet-18 with dynamic entropy leaf expansion)
5. **Stage 5: Automated Paper Artifact Generation** (Vector PDF figures and LaTeX booktabs tables)
6. **Stage 6: Scientific Manuscript Compilation** (Compile `paper/main.pdf` via pdflatex)

---


## Stage 0: Master Configuration & Device Setup

In [1]:
import os
import sys
from pathlib import Path
import torch
import numpy as np
import matplotlib.pyplot as plt

# Ensure project root is in sys.path
root_dir = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(root_dir) not in sys.path:
    sys.path.insert(0, str(root_dir))

# FAST_MODE = True runs a rapid smoke test (~1 minute) across all stages
# Set FAST_MODE = False for full-scale GPU multi-seed benchmark execution
FAST_MODE = True

from src.utils.seed import set_global_seed
set_global_seed(42)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('MCTS-HNM Master Pipeline initialized.')
print('Project root:', root_dir)
print('Compute device:', device)
print('Execution mode:', 'FAST_MODE (Smoke Test / ~1 min)' if FAST_MODE else 'FULL_BENCHMARK (GPU Scale)')


MCTS-HNM Master Pipeline initialized.
Project root: /home/mlima/workspace/mcts-hnm
Compute device: cuda
Execution mode: FAST_MODE (Smoke Test / ~1 min)


## Stage 1: CIFAR-100 Taxonomy and Embedding Verification

In [2]:
from src.data.cifar100_taxonomy import get_cifar100_taxonomy

taxonomy = get_cifar100_taxonomy()
print(f'Loaded CIFAR-100 taxonomy: {len(taxonomy.fine_names)} fine classes across {len(taxonomy.coarse_names)} superclasses.')
for sc_id, sc_name in enumerate(taxonomy.coarse_names[:5]):
    fine_names = [taxonomy.fine_names[f] for f in taxonomy.coarse_to_fine_ids[sc_id]]
    print(f'  Superclass {sc_id:02d} ({sc_name}): {fine_names}')


Loaded CIFAR-100 taxonomy: 100 fine classes across 20 superclasses.
  Superclass 00 (aquatic_mammals): ['beaver', 'dolphin', 'otter', 'seal', 'whale']
  Superclass 01 (fish): ['aquarium_fish', 'flatfish', 'ray', 'shark', 'trout']
  Superclass 02 (flowers): ['orchid', 'poppy', 'rose', 'sunflower', 'tulip']
  Superclass 03 (food_containers): ['bottle', 'bowl', 'can', 'cup', 'plate']
  Superclass 04 (fruit_and_vegetables): ['apple', 'mushroom', 'orange', 'pear', 'sweet_pepper']


## Stage 2: Dual Hierarchy Tree Construction
Building both the Native Semantic Tree and the Data-Driven Ward Confusion Tree:

In [3]:
from src.trees.semantic_tree import build_semantic_tree
from src.trees.confusion_tree import build_confusion_tree

# Generate sample embeddings & index pools for 100 classes
N = 500
d_feat = 512
np.random.seed(42)
train_feats = torch.randn(N, d_feat)
train_labels = np.array([i % 100 for i in range(N)])
val_feats = torch.randn(100, d_feat)
val_labels = np.array([i % 100 for i in range(100)])

class_to_indices = {c: np.where(train_labels == c)[0] for c in range(100)}

# Build Semantic Hierarchy
semantic_tree = build_semantic_tree(class_to_indices=class_to_indices, taxonomy=taxonomy)
semantic_tree.validate_tree()
print(f'Native Semantic Tree built: {semantic_tree.num_nodes} nodes, {semantic_tree.num_leaves} leaves.')

# Build Synthetic/Empirical Confusion Tree
K = 100
conf_matrix = np.eye(K) * 50.0 + np.random.uniform(0.1, 5.0, (K, K))
confusion_tree = build_confusion_tree(conf_matrix, class_to_indices=class_to_indices, taxonomy=taxonomy)
confusion_tree.validate_tree()
print(f'Ward Confusion Tree built: {confusion_tree.num_nodes} nodes, {confusion_tree.num_leaves} leaves.')


Native Semantic Tree built: 121 nodes, 100 leaves.
Ward Confusion Tree built: 199 nodes, 100 leaves.


## Stage 3: Tier-1 MCTS Hard Negative Mining Benchmark

In [4]:
from src.training.trainer import train_scenario1

num_steps = 20 if FAST_MODE else 10000
eval_every = 10 if FAST_MODE else 1000

train_binary = (train_labels == 0).astype(np.int64)
pos_idx = np.where(train_binary == 1)[0]
neg_idx = np.where(train_binary == 0)[0]
val_binary = (val_labels == 0).astype(np.int64)

print(f"Running Tier-1 MCTS training ({num_steps} steps, strategy=mcts_enhanced)...")
res_tier1 = train_scenario1(
    train_features=train_feats,
    train_binary_labels=train_binary,
    positive_indices=pos_idx,
    negative_indices=neg_idx,
    test_features=val_feats,
    test_binary_labels=val_binary,
    strategy="mcts_enhanced",
    tree=semantic_tree,
    num_steps=num_steps,
    eval_every=eval_every,
    pos_batch_size=4,
    neg_batch_size=16,
    device=device,
    seed=42,
)

final_m1 = res_tier1["final_metrics"]
print(f"Tier-1 Training complete. Final minority F1: {final_m1['f1']:.4f}, AUPRC: {final_m1['auprc']:.4f}")


Leaf node 21 (class 4) has only 5 instances (< requested K_mcts=8); sampling with replacement.


Running Tier-1 MCTS training (20 steps, strategy=mcts_enhanced)...


/home/mlima/workspace/mcts-hnm/src/training/trainer.py:300: UserWarning: Leaf node 21 (class 4) has only 5 instances (< requested K_mcts=8); sampling with replacement.
  batch = mcts_sampler.sample_batch(mcts_leaf_id)


Tier-1 Training complete. Final minority F1: 0.0000, AUPRC: 0.0127


## Stage 4: Tier-2 End-to-End CNN Vision Pipeline
Executing trainable ResNet-18 with dynamic prediction-entropy leaf expansion:

In [5]:
from src.training.e2e_trainer import Tier2ResNet18, train_e2e_scenario1

e2e_steps = 15 if FAST_MODE else 5000
train_images = torch.randn(N, 3, 32, 32)
val_images = torch.randn(100, 3, 32, 32)

train_bin_e2e = (train_labels == 0).astype(np.int64)
pos_idx_e2e = np.where(train_bin_e2e == 1)[0]
neg_idx_e2e = np.where(train_bin_e2e == 0)[0]
val_bin_e2e = (val_labels == 0).astype(np.int64)

print(f"Running Tier-2 End-to-End ResNet-18 training ({e2e_steps} steps)...")
res_tier2 = train_e2e_scenario1(
    train_images=train_images,
    train_binary_labels=train_bin_e2e,
    positive_indices=pos_idx_e2e,
    negative_indices=neg_idx_e2e,
    test_images=val_images,
    test_binary_labels=val_bin_e2e,
    strategy="mcts_enhanced",
    tree=semantic_tree,
    num_steps=e2e_steps,
    eval_every=10 if FAST_MODE else 500,
    pos_batch_size=2,
    neg_batch_size=8,
    enable_leaf_expansion=True,
    split_threshold=5 if FAST_MODE else 50,
    width_multiplier=0.25 if FAST_MODE else 1.0,
    device=device,
    seed=42,
)

final_m2 = res_tier2["final_metrics"]
print(f"Tier-2 E2E Training complete. Final F1: {final_m2['f1']:.4f}, AUPRC: {final_m2['auprc']:.4f}")


Running Tier-2 End-to-End ResNet-18 training (15 steps)...
Tier-2 E2E Training complete. Final F1: 0.0000, AUPRC: 0.0182


## Stage 5: Automated Paper Artifact Generation
Generating vector PDF figures and LaTeX booktabs tables:

In [6]:
from src.utils.paper_artifacts import generate_all_paper_artifacts

artifacts = generate_all_paper_artifacts(
    results_dir="results",
    paper_dir="paper",
    metadata_path="paper_metadata.yaml",
)

print(f'Generated {len(artifacts["figures"])} figures and {len(artifacts["tables"])} tables in paper/:')
for tab in artifacts["tables"]:
    print(f'  Table: {tab.name}')
for fig in artifacts["figures"]:
    print(f'  Figure: {fig.name}')


Paper metadata file paper_metadata.yaml not found. Using default PPGC UFRGS metadata.


Generated 6 figures and 3 tables in paper/:
  Table: tab_scenario1_imbalanced.tex
  Table: tab_scenario2_multiclass.tex
  Table: tab_ablation_study.tex
  Figure: fig_convergence_curves.pdf
  Figure: fig_convergence_curves.png
  Figure: fig_tree_visitation_heatmap.pdf
  Figure: fig_tree_visitation_heatmap.png
  Figure: fig_hyperparameter_sensitivity.pdf
  Figure: fig_hyperparameter_sensitivity.png


## Stage 6: LaTeX Manuscript Compilation & Verification

In [7]:
from src.utils.paper_artifacts import compile_paper

try:
    pdf_out = compile_paper(paper_dir="paper")
    print('LaTeX Compilation Succeeded!')
    print(f'Output PDF: {pdf_out} ({pdf_out.stat().st_size:,} bytes)')
except Exception as exc:
    print('Notice:', exc)


Notice: LaTeX document not found at /home/mlima/workspace/mcts-hnm/notebooks/paper/main.tex


## Summary and Reproducibility Verification
All 6 stages of the MCTS-HNM experimental pipeline completed successfully!
